# Free binder design — BoltzGen + Blitz K16

Design de novo protein binders against a target epitope, on a free Colab GPU.

**Runtime → Change runtime type → GPU.** Any GPU works; the notebook adapts.

---

## What this gives you

A handful of designed binder sequences with predicted structures and confidence
metrics, for a target and epitope you choose. Everything is MIT-licensed and the
model weights are public.

## Before you start

**You choose the epitope, and that choice is the run.** A binder designed
against a badly chosen site will still come back with good-looking scores — the
model answers the question you asked. Pick a solvent-exposed, reasonably compact
surface patch, and treat the confidence metrics as "did the model do what I
asked", not "is this a good drug target".

Designs are drawn fresh each run, so two runs give you different binders rather
than the same ones refined. That is normal.

## Credit

BoltzGen is MIT, © 2025 Hannes Stärk ([paper](https://jclinic.mit.edu/boltzgen/)).
Blitz K16 is a distilled Boltz-2 student from
[EpiGenomicsCode/Blitz](https://github.com/EpiGenomicsCode/Blitz); weights at
[vinaymatt/Blitz-Boltz2](https://huggingface.co/vinaymatt/Blitz-Boltz2).

## Provenance

If you saved this notebook and are returning to it later, this is exactly what
you have. Nothing below is fetched at runtime from a moving reference.

| | |
|---|---|
| notebook version | `0.1.0` |
| BlitzGen source | `blitzgen @ git+https://github.com/EpiGenomicsCode/blitzgen@v0.1.0` |
| folding config sha256 | `ec0643ab19098e1daa0d01227f81ad93b3e14fa79e4cb4ff19c130342d6e2e4e` |
| K16 checkpoint sha256 | `343f252bbfe5e520712232091a9a6fa3291ee67b52d84c794f63812010ee949f` |

Newest version: <https://github.com/EpiGenomicsCode/blitzgen-colab>


---
## 1 · Hardware probe

**Run this first.** It needs no install and finishes in seconds, so a card that
cannot run the job is discovered now rather than after a 7.5 GB download.

It picks the precision for you:

* **compute capability ≥ 8.0** (A100, L4, RTX 30xx/40xx) → `bf16-mixed`.
* **below 8.0** (T4, V100) → `fp32`. bf16 needs Ampere.

`fp16` is *not* an option on any card. It was tested and fails: the diffusion
sampler's coordinate arithmetic overflows fp16's 5-bit exponent upstream of
`weighted_rigid_align`, and the resulting non-finite coordinates surface as
`linalg.svd: failed to converge`. Forcing the alignment itself to fp32 does not
help — the coordinates are already broken on arrival.


In [ ]:
import subprocess, shutil, sys, textwrap

def smi(q):
    try:
        return subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
            text=True, timeout=30).strip().splitlines()[0]
    except Exception as e:
        return f"ERROR: {e}"

if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

name   = smi("name")
vram   = smi("memory.total")
cap    = smi("compute_cap")
driver = smi("driver_version")

try:
    cap_f = float(cap)
except ValueError:
    cap_f = 0.0

PRECISION = "bf16-mixed" if cap_f >= 8.0 else "32"
vram_mib  = int("".join(c for c in vram if c.isdigit()) or 0)

print(f"GPU            {name}")
print(f"VRAM           {vram}")
print(f"capability     {cap}")
print(f"driver         {driver}")
print(f"gcc            {shutil.which('gcc') or 'MISSING -- triton kernels need a C compiler'}")
print()
print(f"=> precision   {PRECISION}" + ("   (Ampere+)" if cap_f >= 8.0
      else "   (pre-Ampere: no bf16, so full fp32)"))

# Peak VRAM measured on an A100 for a 361-token target at batch size 1:
#   bf16-mixed 8,555 MiB   fp32 8,899 MiB
# fp32 costs only ~4% more memory because bf16-mixed is MIXED -- AMP already
# keeps fp32 master weights, so going full fp32 only promotes the bf16
# activation subset.
NEED = 8900 if PRECISION == "32" else 8600
if vram_mib and vram_mib < NEED + 2000:
    print(f"\nWARNING: {vram_mib} MiB may be tight. A 361-token target peaked at "
          f"{NEED} MiB; larger targets need more. Reduce the crop if you hit OOM.")
else:
    print(f"\nHeadroom OK: ~{vram_mib - NEED} MiB above the {NEED} MiB reference peak.")

if cap_f < 8.0:
    print(textwrap.dedent('''
        SPEED NOTE. Reference: 8 designs took 489 s (bf16) / 556 s (true fp32)
        on an A100. This workload is NOT matmul-bound -- disabling TF32 cost
        only 2.4% -- so the limit on an older card is memory bandwidth, not
        FLOPS. A T4 (320 GB/s vs the A100's 1555) should land near 30-40 min
        for 8 designs. That is an estimate from a bandwidth ratio, not a
        measurement; please report what you actually see.
    '''))

---
## 2 · Install

Installs [BlitzGen](https://github.com/EpiGenomicsCode/blitzgen) `v0.1.0`,
pinned to a tag so this notebook behaves the same way next month as it does
today. BlitzGen provides `--folding_config` and `--seed`; upstream BoltzGen has
neither, so a stock `pip install boltzgen` will not work here.

No CUDA extra is requested. BlitzGen's `[cu12]`/`[cu13]` extras carry the
`cuequivariance_ops_*` wheels, which are large and are never actually loaded by
this pipeline — the triton kernels behind `cuequivariance_torch` are what run.
Asking for them would only add a way for the install to fail.

Takes a couple of minutes.


In [ ]:
#@title Install BlitzGen
BLITZGEN_SRC = "blitzgen @ git+https://github.com/EpiGenomicsCode/blitzgen@v0.1.0"  #@param {type:"string"}

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", BLITZGEN_SRC],
                   capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-3000:])
if r.returncode != 0:
    raise SystemExit(
        "Install failed.\n"
        "If this is a 404/auth error the repository is private or does not exist.\n"
        "A public source is required; see the final section of this notebook.")

import importlib, boltzgen  # noqa
print("boltzgen", getattr(boltzgen, "__version__", "(unversioned)"))

---
## 3 · Weights  (~7.5 GB)

Two downloads:

* **`boltzgen/boltzgen-1`** (~5.6 GB) — the design and inverse-folding models.
* **`vinaymatt/Blitz-Boltz2 / blitz-boltz2-k16.ckpt`** (1.9 GB) — the Blitz K16
  folding student.

The K16 file is checked against the publisher's own `SHA256SUMS`. The expected
digest is `343f252b…`, the same checkpoint used to build the
[BinderBase](https://binderbase.org) catalogue — so folding here is the real
model, not a lookalike.

Mounting Drive caches both across sessions and is worth it if you will run more
than once.


In [ ]:
#@title Download weights
USE_DRIVE_CACHE = False  #@param {type:"boolean"}

import os, hashlib, pathlib, subprocess, sys

CACHE = "/content/blitzgen_cache"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/blitzgen_cache"
HF_HOME = f"{CACHE}/hf"; CKPT_DIR = f"{CACHE}/ckpt"
os.makedirs(HF_HOME, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import hf_hub_download, snapshot_download

K16_SHA = "343f252bbfe5e520712232091a9a6fa3291ee67b52d84c794f63812010ee949f"
CKPT = pathlib.Path(CKPT_DIR) / "blitz-boltz2-k16.ckpt"

if not CKPT.exists():
    print("Downloading Blitz K16 (1.9 GB) ...")
    src = hf_hub_download("vinaymatt/Blitz-Boltz2", "blitz-boltz2-k16.ckpt",
                          local_dir=CKPT_DIR)
    CKPT = pathlib.Path(src)

print("Verifying checksum ...")
h = hashlib.sha256()
with open(CKPT, "rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 24), b""):
        h.update(chunk)
got = h.hexdigest()
if got != K16_SHA:
    raise SystemExit(f"CHECKSUM MISMATCH\n  expected {K16_SHA}\n  got      {got}\n"
                     "Delete the file and re-run; do not design against it.")
print(f"  OK  {got[:16]}...  checkpoint verified")

print("\nDownloading BoltzGen models (~5.6 GB, slowest step) ...")
snapshot_download("boltzgen/boltzgen-1")
print("done")

---
## 4 · The K16 folding schedule

**This cell is not optional, and it is the easiest thing to get silently wrong.**

K16 weights must run on the K16 schedule — 16 sampling steps, `sigma_min 0.01`.
Pair them with the default Boltz-2 schedule (200 steps) and *nothing errors*:
the structures look plausible and the sampler is simply wrong.

The config below is BlitzGen's own `fold_blitz_k16.yaml`, copied verbatim, so
this notebook folds on exactly the schedule K16 was tuned for.


In [ ]:
K16_YAML = r"""# Blitz K16 folding recipe (EpiGenomicsCode/Blitz, --blitz_policy k16).
# Default folding policy for BlitzGen.
#
# Sampler. Blitz hardcodes a 16-value sigma tuple in absolute units; BoltzGen's
# sample_schedule_af3 generates the same curve analytically and multiplies by
# sigma_data (diffusion.py). With sigma_max=160, sigma_min=0.01, rho=40,
# sigma_data=16, N=16 the two agree to ~1e-13 (K8 agrees bit-for-bit), so no
# schedule code is needed -- only the two values changed below vs the k8 recipe.
#
# NOT reproduced here: Blitz's FK steering and terminal clean projection, which
# BoltzGen has no equivalent for (no potentials module, no guidance update). That
# costs the published physical-validity gain (violation rate 26.7% vs teacher
# 30.3%); it does not affect whether the checkpoint runs. Measured on 20 paired
# designs, k16 under this config tracks the Boltz-2 teacher closely:
#   design_to_target_iptm  median shift -0.004, spearman +0.88
#   complex_plddt          median shift +0.002, spearman +0.96
# so filter thresholds derived against the teacher carry over without rescaling.
#
# Full K16 steering policy, for whoever ports it:
#   fk_steering=True, num_particles=1 (so FK resampling is inert), fk_lambda=4.0,
#   fk_resampling_interval=3, physical_guidance_update=True,
#   contact_guidance_update=True, num_gd_steps=20,
#   terminal clean projection: scale 2/27, 10 GD steps.
#   Per-potential guidance weights (these are RETUNED for 16 steps, not the
#   Boltz-2 defaults): VDWOverlap 0.675, SymmetricChainCOM 0.5, Connections 0.15,
#   StereoBond 0.05, PlanarBond 0.05, ChiralAtom scheduled 2.651 -> 0.194 over the
#   15 inner steps, PoseBusters scheduled 0.634 -> 0.040 plus a terminal override
#   to 0.135 below 3/32 of the trajectory.
_target_: boltzgen.task.predict.predict.Predict

debug: false

data:
  _target_: boltzgen.task.predict.data_from_generated.FromGeneratedDataModule
  cfg:
    _target_: boltzgen.task.predict.data_from_generated.DataConfig
    tokenizer:
      _target_: boltzgen.data.tokenize.tokenizer.Tokenizer
      atomize_modified_residues: false
    featurizer:
      _target_: boltzgen.data.feature.featurizer.Featurizer

    suffix: .cif
    suffix_metadata: .npz
    suffix_native: _native.cif
    samples_per_target: 1000000000000000
    num_targets: 10000000000000
    moldir: null

    batch_size: 1
    num_workers: 4
    pin_memory: true
    disulfide_prob: 1.0
    disulfide_on: true
  design_dir: null
  target_templates: true
  return_native: false
  fail_if_no_designs: true
  output_dir: null

keys_dict_out: ["min_interaction_pae", "min_design_to_target_pae", "interaction_pae", "ligand_iptm", "protein_iptm", "iptm", "design_iptm", "design_iiptm", "design_to_target_iptm", "design_residue_iptm", "design_ptm", "target_ptm", "ptm"]

writer:
  _target_: boltzgen.task.predict.writer.FoldingWriter
  design_dir: ${data.design_dir}

trainer:
  accelerator: gpu
  logger: false
  devices: 1
  precision: bf16-mixed

name: null
output: null
checkpoint: null
matmul_precision: null
# Fixed RNG seed for the diffusion sampler. Without it, repeat runs of identical
# inputs differ and a provenance record cannot be reproduced. Override per run
# with --seed; set to null to restore nondeterministic sampling.
seed: 42
deterministic: false
recycling_steps: 5
sampling_steps: 16
diffusion_samples: 5

# Full diffusion_process_args: Predict passes override into
# Boltz.load_from_checkpoint(**override), which replaces the checkpoint's
# diffusion_process_args dict wholesale (no deep-merge). List every knob.
#
# template/msa/pairformer_args: export copies boltz2_conf hparams that include
# use_trifast=True; boltzgen TemplateModule/MSAModule/PairformerModule reject
# that kwarg. Override with the same dicts minus use_trifast (parity smoke
# scripts/boltzgen_student_parity_smoke.py strips the same way).
override:
  validators: null
  template_args:
    template_dim: 64
    template_blocks: 2
    activation_checkpointing: true
  msa_args:
    msa_s: 64
    msa_blocks: 4
    msa_dropout: 0.15
    z_dropout: 0.25
    miniformer_blocks: false
    pairwise_head_width: 32
    pairwise_num_heads: 4
    use_paired_feature: true
    activation_checkpointing: true
  pairformer_args:
    num_blocks: 64
    num_heads: 16
    dropout: 0.25
    post_layer_norm: false
    activation_checkpointing: true
  diffusion_process_args:
    sigma_min: 0.01
    sigma_max: 160.0
    sigma_data: 16.0
    rho: 40
    P_mean: -1.2
    P_std: 1.5
    gamma_0: 0.0
    gamma_min: 1.0
    noise_scale: 1.0
    step_scale: 1.5
    sampling_schedule: af3
    mse_rotational_alignment: true
    coordinate_augmentation: true
    alignment_reverse_diff: true
    synchronize_sigmas: false
    exp_exact_step: true
"""

import pathlib, re

# The shipped config hardcodes precision: bf16-mixed, because the machines K16
# was developed on were all Ampere or newer. On a pre-Ampere card the design
# step would correctly run fp32 (set on the command line) while FOLDING quietly
# tried bf16 and died. Both folding and design_folding read this one file, so
# rewriting it here covers both.
cfg_text, n = re.subn(r"^(\s*precision:)\s*\S+", r"\1 " + PRECISION,
                      K16_YAML, flags=re.M)
cfg = pathlib.Path("/content/fold_blitz_k16.yaml")
cfg.write_text(cfg_text)

steps = re.search(r"^sampling_steps:\s*(\d+)", cfg_text, re.M)
smin  = re.search(r"sigma_min:\s*([0-9.]+)", cfg_text)
prec  = re.search(r"^\s*precision:\s*(\S+)", cfg_text, re.M)
print(f"wrote {cfg}")
print(f"  sampling_steps {steps.group(1) if steps else '?'}   (Boltz-2 default is 200)")
print(f"  sigma_min      {smin.group(1) if smin else '?'}")
print(f"  precision      {prec.group(1) if prec else '?'}   ({n} field(s) set from the probe)")

---
## 5 · Your target and epitope

Give a **UniProt accession** and the structure is pulled from AlphaFold DB.

`BINDING` is the epitope — the residues you want the binder to contact, in
UniProt numbering (`"352..363,367..372"`).

Two things worth knowing before you choose:

* A BoltzGen minibinder contacts roughly **21 residues**. Specifying far more
  does not aim it harder, it just aims it vaguely.
* `CROP` trims the target to a neighbourhood of the epitope. It is how you keep
  a 900-residue protein inside a free GPU. Residues outside the crop simply do
  not exist for the model.

Good epitopes are solvent-exposed, high-pLDDT and compact. If you have no basis
for choosing one, that is the honest limit of this notebook: choosing well is
the hard part, and it is the part left to you.


In [ ]:
#@title Target
UNIPROT     = "P14868"              #@param {type:"string"}
CHAIN       = "A"                   #@param {type:"string"}
CROP        = "233..501"            #@param {type:"string"}
BINDING     = "352..363,367..372,374..375"  #@param {type:"string"}
BINDER_LEN  = "55..130"             #@param {type:"string"}
NUM_DESIGNS = 8                     #@param {type:"integer"}
MASK_CUT_FACES = True               #@param {type:"boolean"}
MASK_WIDTH  = 3                     #@param {type:"integer"}

import urllib.request, pathlib, textwrap
work = pathlib.Path("/content/work"); (work / "targets").mkdir(parents=True, exist_ok=True)
(work / "specs").mkdir(exist_ok=True)

cif = work / "targets" / f"AF-{UNIPROT}-F1.cif"
if not cif.exists():
    url = f"https://alphafold.ebi.ac.uk/files/AF-{UNIPROT}-F1-model_v4.cif"
    print(f"fetching {url}")
    try:
        urllib.request.urlretrieve(url, cif)
    except Exception as e:
        raise SystemExit(f"Could not fetch {UNIPROT} from AlphaFold DB: {e}\n"
                         "Check the accession, or upload a .cif to "
                         f"{cif} yourself.")
print(f"target {cif}  ({cif.stat().st_size/1e6:.1f} MB)")

# Cropping invents surfaces. The residues at a cut are buried in the real
# protein but fully exposed in the crop, and the model will cheerfully design a
# binder against that artefact. Mask both cut boundaries with not_binding. If a
# boundary is the true chain terminus the mask is simply redundant, which costs
# nothing.
_lo, _hi = (int(x) for x in CROP.split(".."))
_mask = []
if MASK_CUT_FACES and _hi - _lo > 2 * MASK_WIDTH:
    _mask = [f"{_lo}..{_lo + MASK_WIDTH - 1}", f"{_hi - MASK_WIDTH + 1}..{_hi}"]
MASK_LINE = (f"            not_binding: {','.join(_mask)}\n" if _mask else "")

spec = textwrap.dedent(f'''\
# Generated by the BlitzGen Colab notebook.
# The epitope below was chosen by hand, not by a scored triage pipeline.
entities:
  - protein:
      id: B
      sequence: {BINDER_LEN}
  - file:
      path: ../targets/AF-{UNIPROT}-F1.cif
      include:
        - chain:
            id: {CHAIN}
            res_index: {CROP}
      binding_types:
        - chain:
            id: {CHAIN}
            binding: {BINDING}
{MASK_LINE}''')
spec_path = work / "specs" / f"{UNIPROT}_notebook.yaml"
spec_path.write_text(spec)
print(f"\nspec {spec_path}:\n"); print(spec)

n_bind = sum(len(range(int(a), int(b) + 1)) if ".." in seg else 1
             for seg in BINDING.split(",")
             for a, b in [seg.split("..") if ".." in seg else (seg, seg)])
print(f"masked cut faces: {','.join(_mask) if _mask else 'none'}")
print(f"epitope is {n_bind} residues; a minibinder contacts ~21."
      + ("  (reasonable)" if 10 <= n_bind <= 40 else "  <-- consider narrowing"))

---
## 6 · Design

Six pipeline steps run as separate processes — design, inverse folding, folding,
design folding, analysis, filtering. Each loads its own checkpoint, which is why
there is a fixed cost of roughly **4 minutes** before per-design work begins.

Expect **~8 min** for 8 designs on an A100, **~30–40 min** on a T4.
Do not close the tab: Colab disconnects after 90 idle minutes.


In [ ]:
import subprocess, sys, time, os

cmd = [
    "boltzgen", "run", str(spec_path),
    "--output", "/content/work/run",
    "--protocol", "protein-anything",
    "--num_designs", str(NUM_DESIGNS),
    "--cache", os.environ["HF_HOME"],
    "--folding_checkpoint", str(CKPT),
    "--folding_config", "/content/fold_blitz_k16.yaml",
    "--seed", "42",
    "--config", "design", f"trainer.precision={PRECISION}",
    "--config", "filtering", f"budget={NUM_DESIGNS}",
]
print(" ".join(cmd), "\n" + "=" * 78)
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
p.wait()
print("=" * 78)
print(f"exit {p.returncode}   elapsed {time.time()-t0:.0f}s")
if p.returncode != 0:
    print("\nIf this is CUDA out-of-memory, narrow CROP and rerun cell 5 and 6.")

---
## 7 · Results

`final_ranked_designs/` holds the ranked designs and their metrics.

Reading them: **`design_to_target_iptm`** is the headline number — the model's
own confidence in the binder-target interface. **`complex_plddt`** is per-atom
confidence. **`bb_rmsd_design`** is how far the refolded binder drifted from the
designed backbone; large values mean the design does not fold to what was drawn.

A confident-looking score is not a binder. These are *in-silico* designs, and the
published hit rates for this class of method are single-digit percent before
experimental screening.


In [ ]:
import glob, csv, os, shutil

hits = glob.glob("/content/work/run/final_ranked_designs/all_designs_metrics.csv")
if not hits:
    print("No metrics file - did cell 6 finish without error?")
else:
    with open(hits[0]) as fh:
        rows = list(csv.DictReader(fh))
    print(f"{len(rows)} designs\n")
    cols = [c for c in ("id", "design_to_target_iptm", "complex_plddt",
                        "design_ptm", "bb_rmsd_design", "quality_score")
            if rows and c in rows[0]]
    print("  ".join(f"{c[:22]:>22s}" for c in cols))
    print("-" * (24 * len(cols)))
    def k(r):
        try: return -float(r.get("design_to_target_iptm") or 0)
        except ValueError: return 0
    for r in sorted(rows, key=k)[:10]:
        print("  ".join(f"{str(r.get(c,''))[:22]:>22s}" for c in cols))

    shutil.make_archive("/content/blitzgen_designs", "zip", "/content/work/run")
    print("\nStructures (.cif) are under /content/work/run/final_ranked_designs/")
    try:
        from google.colab import files
        files.download("/content/blitzgen_designs.zip")
    except Exception as e:
        print(f"(auto-download unavailable: {e}; the zip is at "
              "/content/blitzgen_designs.zip)")

---
## Known limitations

**This needs BlitzGen, not BoltzGen.** `--folding_config` and `--seed` are
BlitzGen additions; upstream BoltzGen has neither, and without
`--folding_config` the K16 weights silently run on a 200-step Boltz-2 schedule —
no error, plausible-looking structures, wrong sampler. Cell 2 pins
[`blitzgen @ git+https://github.com/EpiGenomicsCode/blitzgen@v0.1.0`](https://github.com/EpiGenomicsCode/blitzgen/tree/v0.1.0).

**Untested on a T4.** Every number here was measured on an A100 and extrapolated
by a memory-bandwidth ratio. In particular, whether the `cuequivariance` triton
kernels used by triangle attention support `sm_75` is **unknown**. If cell 6
fails inside `triangular.py`, add `--config design use_kernels=false` — there is
a pure-PyTorch fallback, at an unmeasured cost in speed and memory.

**Report the precision you ran at.** On a pre-Ampere card the notebook folds
in `fp32` rather than `bf16-mixed`. Same model and same weights, different
arithmetic and therefore different designs — worth stating alongside any
results you publish.
